In [1]:
import numpy as np
import time
from mcts.decoupled_mcts import MCTS
from simulator.mcts_game_state import MCTSGameState
from simulator.mcts_game_state import navigation_rollout

pygame 2.6.1 (SDL 2.28.4, Python 3.13.5)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
from simulator.agents import Robot
from simulator.map import ScenarioMap


num_humans = 5
num_actions = 3
tree_depth = 6
dt = 0.5
human_speed = 1.7

manual_goal = (16, 5)

scenario = ScenarioMap.build_default()
start_position = scenario.cell_to_world((0, 4))

robot = Robot(
    position=start_position,
    theta=0.0,
)

human_positions = np.zeros((num_humans, 2)) + start_position
human_velocities = np.zeros((num_humans, 2))
human_velocities[:, 0] = 1.0
human_orientations = np.arctan2(human_velocities[:, 1], human_velocities[:, 0])
human_goals = human_positions + human_velocities * (human_speed * dt * tree_depth)

positions = np.vstack((robot.position[None, :], human_positions))
orientations = np.concatenate((np.array([robot.theta], dtype=np.float32), human_orientations))
robot_goal = scenario.cell_to_world(manual_goal)[None, :]
goal_positions = np.vstack((robot_goal, human_goals))

num_agents = positions.shape[0]
mcts = MCTS(navigation_rollout, num_agents, num_actions)

root_state = MCTSGameState(
    positions=positions,
    orientations=orientations,
    agent_goal_positions=goal_positions,
    num_actors=num_agents,
    num_actions=num_actions,
    movement_distance=human_speed * dt,
    angle=np.pi / 4.0,
    map=scenario,
    depth=0
)

In [4]:
timings = []
for i in range(2):
    start_time = time.perf_counter()
    _, child_state = mcts.search(root_state, num_simulations=500)
    end_time = time.perf_counter()
    timings.append((end_time - start_time) * 1000)

print(timings)

[117.53231499460526, 115.87004800094292]


In [ ]:

_, child_state = mcts.search(root_state, num_simulations=500)